<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/IntelImageClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
from google.colab import userdata
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms,datasets
from torch.utils.data import DataLoader

In [2]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('activated')

activated


In [3]:
!kaggle datasets download -d puneet6060/intel-image-classification

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
License(s): copyright-authors
100% 346M/346M [00:04<00:00, 73.3MB/s]



In [4]:
!unzip -q intel-image-classification.zip -d ./dataset_folder

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
pin_memory = (True if torch.cuda.is_available() else False)

cpu


In [5]:
train_path = '/content/dataset_folder/seg_train/seg_train'
val_path = '/content/dataset_folder/seg_test/seg_test'

In [10]:
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

In [11]:
train_data = datasets.ImageFolder(root=train_path,transform=train_transform)
val_data = datasets.ImageFolder(root=val_path,transform=val_transform)

In [14]:
train_loader = DataLoader(dataset=train_data,batch_size=32,shuffle=True,num_workers=2,pin_memory=pin_memory)
val_loader = DataLoader(dataset=val_data,batch_size=32,shuffle=True,num_workers=2,pin_memory=pin_memory)

In [15]:
#shape
tempImages,templabels = next(iter(train_loader))
print(templabels.shape)

torch.Size([32])


In [23]:
class CNN(nn.Module):
  def __init__(self) -> None:
    super().__init__()

    self.conv = nn.Sequential(
        nn.Conv2d(3,32,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(32,32,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(64,64,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(128,128,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.flatten = nn.Flatten()

    self.fc = nn.Sequential(
        nn.LazyLinear(128),
        nn.ReLU(),
        nn.Linear(128,6)
    )

  def forward(self,x):
    x = self.conv(x)
    x = self.flatten(x)
    x = self.fc(x)

    return x

In [24]:
model = CNN().to(device)

In [25]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [28]:
#training

epochs = 10

for epoch in range(epochs):

  model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.long().to(device)

    optimizer.zero_grad()

    output = model(images)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()

    preds = torch.argmax(output, dim=1)
    train_correct += (preds == labels).sum().item()
    train_total = labels.size(0)

  train_accuracy = train_correct / train_total

  model.eval()

  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
   for images,labels in val_loader:
    images = images.to(device)
    labels = labels.long().to(device)

    loss.zero_grad()

    output = model(images)
    loss = loss_fn(output,labels)

    val_loss += loss.item()

    valpreds = torch.argmax(output, dim=1)
    val_correct += (valpreds == labels).sum().item()
    val_total = labels.size(0)

  val_accuracy = val_correct / val_total
    # -------- LOG --------
  print(f"Epoch {epoch+1}/{epochs} "
      f"- accuracy: {train_accuracy:.4f} "
      f"- loss: {train_loss/train_total:.4f} "
      f"- val_accuracy: {val_accuracy:.4f}"
      f"- val_loss: {val_loss/val_total:.4f} ")


AttributeError: 'Tensor' object has no attribute 'zero_grad'